# 이진 분류 실험 ④ — 계좌 금액 분포 피처 추가 (amount stats)
1단계 피처 사다리의 세 번째 칸. 03(기본 4 + port 2 + pair 4 + 활동 8) 위에 **이 거래 금액이 그 계좌의 최근 금액 분포에서 어디에 있나** 피처 8개를 더한다. 모델은 GINe 고정, 설정 동일.

| 피처 | 뜻 (전부 그 거래 시각 기준 과거 24시간, 같은 분 제외) |
|---|---|
| `src_amt_mean_1d` / `src_amt_std_1d` / `src_amt_max_1d` | 송신 계좌가 최근 24시간에 보낸 금액의 평균 / 표준편차 / 최대 (ln(1+x)) |
| `src_amt_ratio_1d` | 이 거래 금액이 그 평균의 몇 배인가: ln(1+금액) − ln(1+평균). 이력 없으면 0 |
| `dst_amt_mean_1d` / `dst_amt_std_1d` / `dst_amt_max_1d` / `dst_amt_ratio_1d` | 수신 계좌가 최근 24시간에 받은 금액 쪽 같은 4개 |

03 의 `*_amt_1d` 는 **합계**(활동량)였고, 여기는 **분포**(평소 금액 대비 이상치). 금액 자체(`amount_received`)는 01 부터 있었다.

**가설**: 03 에서 유일하게 나빠진 **UNMATCHED**(패턴 없는 세탁, 양성의 41%) 를 건드리는지가 관전 포인트. 패턴 피처가 아니라 "이 계좌답지 않은 금액" 피처라서, 패턴 recall 은 03 수준을 유지하고 UNMATCHED 의 argmax recall(0.013) · 평균 점수(0.063) 가 오르면 성공. 금액을 쪼개 보내는 STACK · 큰 돈이 한 번에 들어오는 GATHER-SCATTER 도 볼 것.

**메모리** (03 파이프라인 그대로): ① 피처 parquet 은 tx 올리기 전에 ② 피처 float16 으로 붙임 ③ z-norm 행렬은 디스크 memmap ④ 트리 입력은 `rows_matrix`, 테스트 예측은 `predict_rows`.
03 보다 피처 8열이 늘어 트리 입력 X_train 이 9.8GB(float32, 1억 × 24) — 03 방식으로는 RSS 18.5GB 에서 트리 구성 중 죽었다. 그래서 04 부터 트리는 **행렬을 안 만들고** memmap 을 묶음으로 읽어 학습한다 (`train_trees_lowmem`: LightGBM Sequence / XGBoost DataIter, bin 데이터 2~3GB 만 남음). 학습 단계는 memmap 이라 03 과 같음(피크 22GB).

**누수 규칙**: 거래 시각 기준 과거만, 같은 분 제외, z-norm 은 학습 구간 통계. 공통 코드 `bc_lib.py`

# 공통 부분 (작업 준비, 데이터 준비)

## 0. 작업 준비

In [ ]:
import sys, json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [ ]:
# ---- 실험 설정 ------------------------------------------------------------
RUN_NAME = 'amount_stats'         # results/<DATASET>/amount_stats/
PREV_RUN = 'account_activity'     # 비교 기준 (03)
cfg = B.Config(dataset='HI-Large', smoke=False)
B.seed_all(cfg.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = cfg.out_dir(RUN_NAME)

# 앞 단계 피처 (02·03 parquet 재사용) + 이번 단계 피처
PAIR_FEATURES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'reverse_history_count']
PAIR_PATH = cfg.feat_dir / 'pair_history.parquet'
ACT_FEATURES = ['src_out_1h', 'src_out_1d', 'src_uniq_dst_1d', 'src_out_amt_1d', 'dst_in_1h', 'dst_in_1d', 'dst_uniq_src_1d', 'dst_in_amt_1d']
ACT_PATH = cfg.feat_dir / 'account_activity.parquet'
STAGE_FEATURES = ['src_amt_mean_1d', 'src_amt_std_1d', 'src_amt_max_1d', 'src_amt_ratio_1d',
                  'dst_amt_mean_1d', 'dst_amt_std_1d', 'dst_amt_max_1d', 'dst_amt_ratio_1d']
STAGE_PATH = cfg.feat_dir / 'amount_stats.parquet'
PREV_FEATURES = PAIR_FEATURES + ACT_FEATURES
GNN_EDGE_FEATURES = B.BASIC_EDGE_FEATURES + B.PORT_FEATURES + PREV_FEATURES + STAGE_FEATURES   # FEAT16 열 순서 (26). port 는 4,5 번
TREE_FEATURES = B.BASIC_EDGE_FEATURES + PREV_FEATURES + STAGE_FEATURES                         # 트리는 port 없이 (24)
for p_ in (PAIR_PATH, ACT_PATH):
    assert p_.exists(), f'앞 단계 parquet 이 없다: {p_}'
DEVICE, OUT_DIR, len(GNN_EDGE_FEATURES), len(TREE_FEATURES)

In [ ]:
cfg

## 1. 이번 단계 피처 계산 — 계좌 금액 분포 (tx 를 올리기 **전에**)
03 과 같은 이유: DuckDB 창 함수(12GB) 와 tx 를 동시에 올리면 한도를 넘는다. 한 번 만들면 이 절은 바로 지나간다

- 03 의 창과 같은 꼴(`RANGE … 24 hours PRECEDING AND 1 minute PRECEDING`) 에 집계만 `avg / stddev_pop / max` 로 바꾼 것. src 파티션 한 번, dst 파티션 한 번
- `ratio` 는 저장 단계에서 `ln(1 + received) − ln(1 + avg)` 로 계산. 이력이 없으면(avg NULL) 0 — 03 의 `*_out_1d = 0` 이 "이력 없음" 을 이미 말해 주므로 여기서는 중립값
- 금액은 `received` (Amount Received). 결과는 `features_bc/amount_stats.parquet` (edge_id 순)

In [ ]:
con = L.connect(threads=8, memory='12GB')
currency_codes, format_codes = B.code_tables(con, cfg)
PORTS_PATH = B.ensure_ports(con, cfg)               # 있으면 건너뜀

In [ ]:
# 이전 실행이 COPY 도중 죽었으면 불완전한 parquet 이 남아 있을 수 있다 → 행 수가 tx 와 다르면 지우고 다시 만든다
if STAGE_PATH.exists():
    n_stage = con.execute(f"SELECT count(*) FROM read_parquet({L.q(STAGE_PATH)})").fetchone()[0]
    n_tx = con.execute(f"SELECT count(*) FROM read_parquet({L.q(cfg.tx_path)})").fetchone()[0]
    if n_stage != n_tx:
        print(f'불완전한 {STAGE_PATH.name} ({n_stage:,} / {n_tx:,} 행) → 삭제 후 재생성'); STAGE_PATH.unlink()
if not STAGE_PATH.exists():
    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE tx_min AS
        SELECT edge_id, src, dst, ts, received FROM read_parquet({L.q(cfg.tx_path)})""")
    # (a) 송신 계좌 기준 — 최근 24시간에 보낸 금액의 분포
    con.execute("""
        CREATE OR REPLACE TABLE src_win AS
        SELECT edge_id, received,
               avg(received)        OVER w1d AS amt_mean,
               stddev_pop(received) OVER w1d AS amt_std,
               max(received)        OVER w1d AS amt_max
        FROM tx_min
        WINDOW w1d AS (PARTITION BY src ORDER BY ts RANGE BETWEEN INTERVAL '24 hours' PRECEDING AND INTERVAL '1 minute' PRECEDING)""")
    print(f'  src_win {time.time()-t0:.0f}s')
    # (b) 수신 계좌 기준 — 최근 24시간에 받은 금액의 분포
    con.execute("""
        CREATE OR REPLACE TABLE dst_win AS
        SELECT edge_id,
               avg(received)        OVER w1d AS amt_mean,
               stddev_pop(received) OVER w1d AS amt_std,
               max(received)        OVER w1d AS amt_max
        FROM tx_min
        WINDOW w1d AS (PARTITION BY dst ORDER BY ts RANGE BETWEEN INTERVAL '24 hours' PRECEDING AND INTERVAL '1 minute' PRECEDING)""")
    print(f'  dst_win {time.time()-t0:.0f}s')
    # (c) ln 변환 + ratio 계산 후 저장 (edge_id 순). 이력 없음(NULL) → mean/std/max 0, ratio 0
    con.execute(f"""
        COPY (
            SELECT s.edge_id,
                   ln(1 + coalesce(s.amt_mean, 0))::FLOAT                                                        AS src_amt_mean_1d,
                   ln(1 + coalesce(s.amt_std, 0))::FLOAT                                                         AS src_amt_std_1d,
                   ln(1 + coalesce(s.amt_max, 0))::FLOAT                                                         AS src_amt_max_1d,
                   (CASE WHEN s.amt_mean IS NULL THEN 0 ELSE ln(1 + s.received) - ln(1 + s.amt_mean) END)::FLOAT AS src_amt_ratio_1d,
                   ln(1 + coalesce(d.amt_mean, 0))::FLOAT                                                        AS dst_amt_mean_1d,
                   ln(1 + coalesce(d.amt_std, 0))::FLOAT                                                         AS dst_amt_std_1d,
                   ln(1 + coalesce(d.amt_max, 0))::FLOAT                                                         AS dst_amt_max_1d,
                   (CASE WHEN d.amt_mean IS NULL THEN 0 ELSE ln(1 + s.received) - ln(1 + d.amt_mean) END)::FLOAT AS dst_amt_ratio_1d
            FROM src_win s JOIN dst_win d USING (edge_id)
            ORDER BY s.edge_id
        ) TO {L.q(STAGE_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.execute("DROP TABLE src_win; DROP TABLE dst_win; DROP TABLE tx_min")
    print(f'{STAGE_PATH.name} 생성 {time.time()-t0:.0f}s')
con.execute(f"SELECT count(*), {', '.join(f'avg({c}) AS avg_{c}' for c in STAGE_FEATURES)} FROM read_parquet({L.q(STAGE_PATH)})").df().T

In [ ]:
# 창 동작 확인 — 하루에 거래 8건쯤 보낸 계좌 하나의 송금 타임라인. mean/max 가 과거 24시간 금액으로만 움직이고, ratio 가 (이번 금액 vs 평균) 인지
# 주의: 1.8억 건끼리 조인하지 않는다 → 그 계좌의 edge_id 만 골라 피처 parquet 에서 읽는다
acct = con.execute(f"SELECT src FROM read_parquet({L.q(cfg.tx_path)}) GROUP BY src, day ORDER BY abs(count(*) - 8), src LIMIT 1").fetchone()[0]
ids = con.execute(f"SELECT edge_id, ts, dst, received FROM read_parquet({L.q(cfg.tx_path)}) WHERE src = {acct} ORDER BY ts LIMIT 40").df()
feat = con.execute(f"SELECT edge_id, src_amt_mean_1d, src_amt_max_1d, src_amt_ratio_1d FROM read_parquet({L.q(STAGE_PATH)}) WHERE edge_id IN ({', '.join(map(str, ids.edge_id))})").df()
tl = ids.merge(feat, on='edge_id').sort_values('ts')
tl['mean_1d (금액)'] = np.expm1(tl['src_amt_mean_1d']).round(2); tl['max_1d (금액)'] = np.expm1(tl['src_amt_max_1d']).round(2)   # ln(1+x) 를 되돌려 금액으로
tl['ratio (배)'] = np.exp(tl['src_amt_ratio_1d']).round(2)
tl

## 2. 데이터 준비 — 거래 전체 + port + 앞 단계 피처(02·03) + 이번 피처
전부 edge_id 순 parquet 이라 조인 없이 열만 붙인다 (`attach_parquet`, float16)

In [ ]:
con.close(); con = L.connect(threads=8, memory='3GB')      # 창 함수가 잡아 둔 DuckDB 버퍼 반납. 이후 쿼리는 작아서 3GB 면 충분
con.register('currency_codes', currency_codes); con.register('format_codes', format_codes)   # 코드표는 연결마다 다시 등록
tx = B.load_tx_lowmem(con, cfg, port_path=PORTS_PATH)                      # 기본 9열 + port 2열 — parquet 에서 열 하나씩
B.trim('(tx 기본+port 적재 후)')
tx = B.attach_parquet(tx, PAIR_PATH, PAIR_FEATURES, dtype=np.float16)     # 02
tx = B.attach_parquet(tx, ACT_PATH, ACT_FEATURES, dtype=np.float16)       # 03
tx = B.attach_parquet(tx, STAGE_PATH, STAGE_FEATURES, dtype=np.float16)   # 이번 피처
B.trim('(피처 20열 붙인 후)'); tx.shape

In [ ]:
tx.head()

In [ ]:
tx[STAGE_FEATURES].describe().T

In [ ]:
# 라벨별 평균 — 세탁 거래에서 값이 다른가 (피처가 신호를 갖는지 미리 확인). pandas groupby 는 열 단위라 가볍다
tx.groupby('y')[STAGE_FEATURES].mean().T.rename(columns={0: 'normal', 1: 'laundering'})

## 3. z-norm (학습 구간 통계) + 표본

In [ ]:
in_train = B.window_mask(tx, cfg.train_window)
FEAT_PATH = cfg.feat_dir / f'feat16_{RUN_NAME}.npy'                       # z-norm 행렬을 디스크 memmap 으로 (26열 9.3GB 가 RSS 에서 빠짐)
feat_z, znorm_table = B.znorm_features(tx, GNN_EDGE_FEATURES, in_train, out_path=FEAT_PATH)   # tx 에서 피처 열은 떨어짐
B.trim('(z-norm 후)'); znorm_table

In [ ]:
print(f"데이터 기간 {tx['day'].min().date()} ~ {tx['day'].max().date()}")
for name, w in [('train', cfg.train_window), ('val', cfg.val_window), ('test', cfg.test_window)]:
    m = B.window_mask(tx, w); assert m.sum() > 0, f'{name} 구간 {w} 에 거래가 없음'
    print(f'{name:5s} {w}  거래 {m.sum():>12,}  세탁 {tx.loc[m, "y"].sum():>8,}  비율 {tx.loc[m, "y"].mean()*100:.3f}%')

In [ ]:
val_df = B.val_sample(con, cfg.val_window, cfg.val_neg_rate)        # 검증: 가중 표본 (05 공유)
test_df = B.test_full(con, cfg.test_window)                         # 테스트: 구간 전체
in_degree = B.recipient_indegree(con, cfg)                           # 분석용
con.close()

all_edge_ids = tx['edge_id'].to_numpy()
val_idx = B.rows_of(all_edge_ids, val_df['edge_id'].to_numpy()); test_idx = B.rows_of(all_edge_ids, test_df['edge_id'].to_numpy())
assert (tx['y'].to_numpy()[val_idx] == val_df['y'].to_numpy()).all()
del all_edge_ids
val_df.shape, test_df.shape

# 트리 모델 (기본 4 + pair 4 + 활동 8 + 이번 피처 8)

## 4. LightGBM / XGBoost

In [ ]:
results = {}
tree_cols = [GNN_EDGE_FEATURES.index(c) for c in TREE_FEATURES]
train_mask = in_train; train_rows = np.flatnonzero(train_mask)
# X_train(1억 × 24 float32 = 9.8GB) 을 만들지 않는다 — 03 방식(rows_matrix)으로는 RSS 18.5GB 에서 트리 구성 중 한도 초과.
# LightGBM 은 Sequence, XGBoost 는 DataIter 로 memmap 을 200만 행씩 읽어 bin 만 남긴다 (train_trees_lowmem). 검증 표본만 행렬로
y_train = tx['y'].to_numpy()[train_mask]
X_val, y_val, w_val = B.rows_matrix(feat_z, val_idx, tree_cols), val_df['y'].to_numpy(), val_df['w'].to_numpy()
test_pred = lambda m: B.predict_rows(m, feat_z, test_idx, tree_cols)     # 테스트 행렬(2GB)도 안 만들고 묶음 예측
B.trim('(트리 입력 준비 후)'); len(train_rows), X_val.shape

In [ ]:
lgb_model, xgb_model = B.train_trees_lowmem(feat_z, train_rows, tree_cols, y_train, X_val, y_val, w_val, test_pred, val_df, test_df, cfg, results)
pd.DataFrame({'feature': TREE_FEATURES, 'lightgbm_gain': lgb_model.booster_.feature_importance('gain'),
              'xgboost_gain': [xgb_model.get_booster().get_score(importance_type='gain').get(f'f{i}', 0.0) for i in range(len(TREE_FEATURES))]})

In [ ]:
# 트리 테스트 점수도 저장 (분석용) — 묶음 예측
for name, model in [('lightgbm', lgb_model), ('xgboost', xgb_model)]:
    pd.DataFrame(dict(edge_id=test_df.edge_id, y=test_df.y, score=test_pred(model))).to_parquet(OUT_DIR / f'{name}_test_scores.parquet', index=False)
del y_train, X_val; B.trim('(트리 입력 삭제 후)')
pd.DataFrame({k: v['val'] for k, v in results.items()}).T.round(4)

# GNN (GINe, 01 과 같은 설정)

## 5. 그래프 — 한 번에 하나만 (bc_lib.GraphStore)

In [ ]:
# feat_z 가 이미 memmap 이라 GraphStore 는 그대로 쓴다. 02 는 이 시점 RSS 약 13GB, 학습 중 20.7GB 였다 — 여기선 피처 행렬이 빠져 더 작아야 정상
store = B.GraphStore(tx, feat_z, cfg, train_rows=np.flatnonzero(train_mask), val_idx=val_idx, test_idx=test_idx,
                     port_cols=[GNN_EDGE_FEATURES.index('in_port'), GNN_EDGE_FEATURES.index('out_port')])
del tx, feat_z; B.trim('(tx, feat_z 삭제 후)')
store.edge_dim

In [ ]:
# 배치 하나 꺼내서 모양 확인 — 2048 target 이 서브그래프 하나를 공유, 엣지 피처 26열
batch = next(iter(store.use('train')))
print('노드', batch['node'].num_nodes, ' to 간선', batch[B.TO].num_edges, ' rev_to 간선', batch[B.REV].num_edges,
      ' target', batch[B.TO].edge_label_index.shape[1], ' 양성', int(batch[B.TO].edge_label.sum()),
      ' x 열', batch['node'].x.shape[1], ' edge_attr 열', batch[B.TO].edge_attr.shape[1], ' target_attr', tuple(batch.target_attr.shape))

In [ ]:
# 모델 모양 확인
model = B.make_gine(store.edge_dim, cfg, edge_updates=False).to(DEVICE)
logits, labels = B.forward_batch(model, batch, DEVICE)
print('파라미터', sum(p.numel() for p in model.parameters()), ' 로짓', tuple(logits.shape))
del model, batch

## 6. 학습 — GINe

In [ ]:
gine_model, gine_hist = B.train_gnn('gine', store, cfg, val_df, OUT_DIR, DEVICE, edge_updates=False, results=results)
results['gine']['val']

In [ ]:
# 학습 곡선
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
hv = gine_hist.dropna(subset=['AP']); ax[0].plot(hv['epoch'], hv['AP'], marker='o'); ax[1].plot(gine_hist['epoch'], gine_hist['train_loss'])
ax[0].set_title('val AP'); ax[1].set_title('train loss'); ax[0].set_xlabel('epoch'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(OUT_DIR / 'curves.png', dpi=120); plt.show()

## 7. 테스트 — 구간 전체

In [ ]:
gine_score = B.test_gnn('gine', store, cfg, test_df, OUT_DIR, DEVICE, edge_updates=False, results=results)

# 결과 — 저장된 점수 파일만으로 돌아간다
커널을 새로 켰으면 **0 절(설정 셀 3개)까지만** 돌리고 바로 여기부터. 재료는 `results/<DATASET>/<RUN_NAME>/`
- `{model}_test_scores.parquet` (edge_id, y, score) — 7 절·4 절에서 저장
- `results.json` — 검증/테스트 지표 (`results` dict 복구용)
- dst 와 수신자 in-degree 만 DuckDB 에서 다시 읽는다 (1~2 분)

운영점(컷)은 두 종류: **예산 컷** `r@b%` = 테스트 전체 상위 b% 를 알림으로 (0.05 / 0.1 / 0.5 / 1 / 2 / 5 / 10 % / all),
**argmax** = 확률 0.5 (논문 F1 운영점, Multi-GNN 부록 Table 13 의 패턴별 recall 과 같은 기준). 0.1% ≈ 양성 수 규모 ≈ argmax 와 비슷한 알림 수.

## 8. 준비 — 점수 · 라벨 · in-degree

In [ ]:
import importlib; importlib.reload(B)
MODELS = [n for n in ['lightgbm', 'xgboost', 'gine'] if (OUT_DIR / f'{n}_test_scores.parquet').exists()]
scores = {n: B.load_scores(OUT_DIR / f'{n}_test_scores.parquet') for n in MODELS}
if 'test_df' not in globals() or 'in_degree' not in globals():
    con = L.connect(threads=8, memory='12GB')
    test_df = B.test_full(con, cfg.test_window); in_degree = B.recipient_indegree(con, cfg); con.close()
for n in MODELS:
    assert (scores[n]['edge_id'].to_numpy() == test_df['edge_id'].to_numpy()).all(), f'{n} 점수와 라벨 순서 불일치'
if 'results' not in globals() or not results:
    results = json.load(open(OUT_DIR / 'results.json'))['results']
print(MODELS, '| 테스트 간선', f"{len(test_df):,}", '| 양성', f"{int(test_df.y.sum()):,}", '| 컷', list(B.cut_thresholds(scores[MODELS[0]]['score'].to_numpy())))

## 9. 운영점별 precision / recall
같은 점수를 컷만 바꿔 가며 본다. 예산을 줄이면 precision 이 오르고 recall 이 떨어지는 그 교환을 한 표에.

In [ ]:
# 컷마다 임계값 · 알림 수 · TP/FP/FN · precision · recall (모델별). 'all' = 전부 알림 (precision = 양성 비율)
cut_df = pd.concat([B.cut_table(scores[n]).assign(model=n) for n in MODELS])
cut_df.to_csv(OUT_DIR / 'analysis_cuts.csv', encoding='utf-8-sig')
cut_df[cut_df.model == 'gine']

In [ ]:
# 모델 나란히 — 컷별 recall / precision
cut_df.reset_index().pivot(index='cut', columns='model', values=['recall', 'precision']).loc[cut_df.index.unique()]

In [ ]:
# PR 곡선 — 컷을 연속으로 훑은 것. AP = 곡선 아래 면적. 세로선 = GINe argmax 운영점
B.plot_pr(scores, path=OUT_DIR / 'pr_curve.png', mark=(cut_df.loc['argmax'].query("model == 'gine'")['recall'].iloc[0], 'GINe argmax'))

In [ ]:
# 혼동행렬 — GINe, 컷 4개 나란히. 색 = 행 비율 (세탁 행 오른쪽 칸 = recall), 숫자 = 건수
B.plot_confusion_grid(scores['gine'], cuts=('r@0.1%', 'r@1%', 'r@10%', 'argmax'), title='GINe (pair_history)', path=OUT_DIR / 'confusion_cuts.png')

## 10. 패턴별 — 어느 패턴을 어느 컷에서 놓치나
패턴 안의 거래는 모두 양성이라 recall 만 본다. 예산을 줄일 때 먼저 떨어지는 패턴이 "모델이 덜 확신하는" 패턴.

In [ ]:
# 패턴 × 컷 recall (모델별 저장, GINe 표시). 'recall' 열 = 1% (01 과 호환), mean_score = 그 패턴 양성의 평균 점수
pattern_df = pd.concat([B.pattern_recall(scores[n]).assign(model=n) for n in MODELS])
pattern_df.to_csv(OUT_DIR / 'analysis_pattern.csv', encoding='utf-8-sig')
CUT_COLS = [c for c in pattern_df.columns if c.startswith('r@') or c in ('all', 'argmax')]
pattern_df[pattern_df.model == 'gine'][['n'] + CUT_COLS + ['mean_score']]

In [ ]:
# 같은 표를 그림으로 — 행 = 패턴, 열 = 컷. 왼쪽(빡빡한 예산)에서 먼저 하얘지는 행이 약한 패턴
B.plot_heatmap(pattern_df[pattern_df.model == 'gine'][CUT_COLS], title='GINe recall: pattern x cut', path=OUT_DIR / 'pattern_x_cut.png')

In [ ]:
# 놓친 세탁(FN) 의 구성 — 컷마다 FN 이 어느 패턴에서 나오나. FN% 가 pos_share 보다 크면 그 패턴을 유독 놓침
groups = dict(pattern=B.pattern_array(scores['gine']), degree_bin=B.degree_bin_of(test_df['dst'].to_numpy(), in_degree))
cb = B.confusion_breakdown(scores['gine'], groups)
for axis in groups:
    cb['fn'][axis].to_csv(OUT_DIR / f'analysis_fn_{axis}.csv', encoding='utf-8-sig'); cb['fp'][axis].to_csv(OUT_DIR / f'analysis_fp_{axis}.csv', encoding='utf-8-sig')
cb['fn']['pattern'][['pos_share'] + [c for c in cb['fn']['pattern'].columns if c.startswith('FN% ')]]

## 11. 수신자 in-degree 층화 (SALT-GNN 프로토콜)
허브(차수 큰 수신자) 쪽 거래가 약점인지. AP/F1 은 구간 안에서 따로 계산, recall/precision 은 전체 컷 임계값을 구간에 적용.

In [ ]:
# 구간별 AP · F1 · 양성 비율 (모델별 저장, GINe 표시)
degree_df = pd.concat([B.degree_stratified(scores[n], test_df['dst'].to_numpy(), in_degree).assign(model=n) for n in MODELS])
degree_df.to_csv(OUT_DIR / 'analysis_degree.csv', encoding='utf-8-sig')
degree_df[degree_df.model == 'gine']

In [ ]:
# 구간 × 컷 — recall 과 precision (GINe). 같은 임계값인데 구간마다 precision 이 다르면 알림이 그쪽에 몰리는 것
deg_cut = B.group_by_cut(scores['gine'], groups['degree_bin'])
order = [b for b in [f'[{lo}-{hi}]' if hi < 10**9 else f'[{lo}+]' for lo, hi in B.DEGREE_BINS] if b in deg_cut['recall'].index]
deg_cut['recall'].loc[order].to_csv(OUT_DIR / 'analysis_degree_x_cut_recall.csv', encoding='utf-8-sig')
deg_cut['precision'].loc[order].to_csv(OUT_DIR / 'analysis_degree_x_cut_precision.csv', encoding='utf-8-sig')
display(deg_cut['recall'].loc[order]); deg_cut['precision'].loc[order]

In [ ]:
# 잘못 잡은 정상(FP) 이 어느 차수 구간에 몰리나 — FP% 가 neg_share 보다 크면 그 구간에 알림이 과함
cb['fp']['degree_bin'][['neg_share'] + [c for c in cb['fp']['degree_bin'].columns if c.startswith('FP% ')]]

## 12. 앞 단계 (PREV_RUN) 와 비교
앞 단계 결과 폴더의 `analysis_*.csv` 와 비교. 01 은 모델 이름이 `gine_basic`, 02 부터는 `gine`.

In [ ]:
# 패턴별 recall — 1% 와 argmax 두 컷에서 앞 단계 GINe vs 이번 GINe
PREV_GNN = 'gine_basic' if PREV_RUN == 'basic_features' else 'gine'
prev_path = Path.cwd() / 'results' / cfg.dataset / PREV_RUN / 'analysis_pattern.csv'
if prev_path.exists():
    prev = pd.read_csv(prev_path, index_col=0); prev = prev[prev.model == PREV_GNN]; cur = pattern_df[pattern_df.model == 'gine']
    cmp = pd.DataFrame({f'{PREV_RUN} r@1%': prev['recall'], f'{RUN_NAME} r@1%': cur['recall']})
    cmp['diff r@1%'] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
    if 'argmax' in prev.columns:
        cmp[f'{PREV_RUN} argmax'] = prev['argmax']; cmp[f'{RUN_NAME} argmax'] = cur['argmax']; cmp['diff argmax'] = cur['argmax'] - prev['argmax']
    display(cmp.loc[[p for p in B.PATTERN_ORDER + ['ALL'] if p in cmp.index]].round(4))
else:
    print('앞 단계 분석표 없음 —', prev_path)

In [ ]:
# 수신 계좌 degree 구간별 AP — 허브 구간이 달라졌나
prev_path = Path.cwd() / 'results' / cfg.dataset / PREV_RUN / 'analysis_degree.csv'
if prev_path.exists():
    prev_d = pd.read_csv(prev_path, index_col=0); prev_d = prev_d[prev_d.model == PREV_GNN]; cur_d = degree_df[degree_df.model == 'gine']
    display(pd.DataFrame({f'{PREV_RUN} AP': prev_d['AP'], f'{RUN_NAME} AP': cur_d['AP'], 'diff': cur_d['AP'] - prev_d['AP'],
                          f'{PREV_RUN} F1': prev_d['F1'], f'{RUN_NAME} F1': cur_d['F1'], 'n_pos': cur_d['n_pos']}).round(4))
else:
    print('앞 단계 분석표 없음:', prev_path)

## 13. 비교표 + 저장

In [ ]:
compare_df = B.save_results(OUT_DIR, cfg, results, features=dict(tree=TREE_FEATURES, gnn=GNN_EDGE_FEATURES))
compare_df

In [ ]:
# 앞 단계 전체 지표와 나란히
prev_c = pd.read_csv(Path.cwd() / 'results' / cfg.dataset / PREV_RUN / 'compare.csv', index_col=[0, 1])
pd.concat([prev_c.assign(run=PREV_RUN), compare_df.assign(run=RUN_NAME)]).reset_index().set_index(['run', 'model', 'split'])[['AP', 'F1', 'F1_at_0.5', 'recall_at_1pct', 'unmatched_recall_at_1pct']]

In [ ]:
sorted(p.name for p in OUT_DIR.iterdir())   # 이 폴더가 report.md 의 재료

## 14. 다음
- 05: GFP 모티프 (사이클·scatter-gather 참여 수, snapml GraphFeaturePreprocessor) → CYCLE, SCATTER-GATHER
- v10 36피처 참조 런 → 1단계 정리 (피처 사다리 표 + UNMATCHED 추이)
- 서비스 프로토콜 테스트 (`test_gnn_daily`: 그날 끝까지 그래프 + 그날 target) 를 02·03·04 체크포인트로